# ECG Liquid Network Analysis

End-to-end workflow: data validation, training, imbalance experiments, validation-only decision tuning, and test evaluation.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data.dataset import load_csv_arrays
from src.data.validation import validate_ecg_csv
from src.utils.config import load_config, ensure_output_dirs
from src.utils.paths import resolve_path
from src.utils.seed import set_seed

config = load_config(ROOT / "src" / "config" / "default.yaml")
ensure_output_dirs(config)
set_seed(config["project"]["seed"])
class_names = {int(k): v for k, v in config["classes"].items()}
paths = config["paths"]

## 1. Data validation and class imbalance

In [ ]:
for rel in [paths["train_csv"], paths["test_csv"]]:
    issues = validate_ecg_csv(resolve_path(rel), config["data"]["num_features"], config["data"]["num_classes"])
    print(rel, "OK" if not issues else issues)

train_x, train_y = load_csv_arrays(paths["train_csv"])
test_y = load_csv_arrays(paths["test_csv"])[1]
print("Train class counts:\n", pd.Series(train_y).value_counts().sort_index().rename(index=class_names))
print("\nTest class counts:\n", pd.Series(test_y).value_counts().sort_index().rename(index=class_names))

## 2. Evaluation protocol (no test leakage)

See `reports/summaries/evaluation_protocol.md`.

- Train on the train split; tune checkpoints, class-weight/sampling choices, and logit biases on the **validation** split only.
- Select the final experiment using **validation macro-F1** (secondary: min validation per-class F1, validation macro ROC-AUC) via `scripts/build_experiment_table.py` → `reports/summaries/model_selection.json`.
- Freeze the selected model and validation-derived biases; run the **test** set once for `reports/summaries/final_test_report.json`.
- Earlier test metrics kept in experiment files are **diagnostic/reference** only and must not be used for selection.

In [ ]:
comparison = pd.read_csv(resolve_path("reports/tables/experiment_comparison.csv"))
selection = json.loads(resolve_path("reports/summaries/model_selection.json").read_text(encoding="utf-8"))
final_report_path = resolve_path("reports/summaries/final_test_report.json")
if final_report_path.exists():
    final_report = json.loads(final_report_path.read_text(encoding="utf-8"))
    print("Deployed model (final test report):", final_report.get("selected_experiment"))
print("Selected (validation):", selection.get("selected_experiment"))
comparison

## 3. Train or run systematic experiments

Full experiment grid:

```bash
python scripts/run_experiments.py
```

Single experiment example:

```bash
python scripts/run_experiments.py --only stride2_hidden128_weighted_ce
```

In [ ]:
# Optional: run default training pipeline
# %run ../scripts/train.py

## 4. Validation-only logit bias (freeze before test)

In [ ]:
# %run ../scripts/apply_val_bias_checkpoint.py
# Optional single test report: add --write-test-report

## 5. Final test metrics and figures

In [ ]:
final_path = resolve_path("reports/summaries/final_test_report.json")
if final_path.exists():
    final_report = json.loads(final_path.read_text(encoding="utf-8"))
    metrics = final_report["test_metrics"]
    print(final_report.get("note", ""))
else:
    metrics = json.loads(resolve_path("reports/metrics/test_metrics.json").read_text(encoding="utf-8"))
print("Macro-F1:", metrics["macro_f1"])
print("Macro precision:", metrics["macro_precision"])
print("Macro recall:", metrics["macro_recall"])
print("Macro ROC-AUC:", metrics["macro_roc_auc"])
print(metrics["classification_report"])

fig_paths = [
    "graphs/dataset/class_distribution.png",
    "graphs/evaluation/confusion_matrix.png",
    "graphs/evaluation/confusion_matrix_normalized.png",
    "graphs/evaluation/roc_curves.png",
    "graphs/evaluation/precision_recall_curves.png",
    "graphs/training/loss_curves.png",
    "graphs/explainability/integrated_gradients_per_class.png",
]
for rel in fig_paths:
    p = resolve_path(rel)
    if p.exists():
        plt.figure(figsize=(8, 4))
        plt.imshow(plt.imread(p))
        plt.axis("off")
        plt.title(p.name)
        plt.show()